# P7 / 01: capitalisation ranks, rank-space residuals and rolling network fits

Protocol: [`001_claude_p7_rank_space_preregistration.md`](../../discussions/Rank_Space/001_claude_p7_rank_space_preregistration.md).

- **Capitalisation:** split-adjusted (not dividend-adjusted) price × point-in-time SEC shares outstanding, on today's split basis.
- **Ranks:** 1–400 each close.
- **Rank returns:** the change in the k-th largest cap. These are not tradable.
- **Residuals:** PCA with 1 factor.
- **Network:** P6's CNN + Transformer.
- **Two arms:** `RS-paper` (gross mean-variance on rank returns) and `RS-net` (Sharpe of the tradable rank-mapped book net of measured costs).
- **Fits:** rolling 750 / 250 / 125, 3 seeds. Every fit is cached.

In [1]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p6, p7, s04, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years, session_schedule
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P7_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p7.P7Config()
ARMS = {"RS-paper": "paper", "RS-net": "net"}
CACHE = work_dir() / "results" / ("p7_synthetic" if SYNTHETIC else "p7")
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, n_ranks=80, train_days=300, valid_days=100, first_test="2017-01-03", end="2017-12-29",
                  seeds=(0,), epochs=3, patience=3, n_placebo=50, n_boot=2000)
import torch
from equity_trading import p7_model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " -- CUDA not available"))
print(f"synthetic={SYNTHETIC} | ranks {CFG.n_ranks} | arms {list(ARMS)} | seeds {CFG.seeds}")

torch 2.14.0+cu126 | device cuda (NVIDIA GeForce GTX 1070)
synthetic=False | ranks 400 | arms ['RS-paper', 'RS-net'] | seeds (0, 1, 2)


## 1. Returns, split-only prices, shares outstanding, capitalisation, costs

In [2]:
t0 = time.time()
if SYNTHETIC:
    R, CAP = p7.synthetic_market(n_days=1800, n_stocks=120, start="2011-01-03")
    COST = pd.DataFrame(2e-4, index=R.index, columns=R.columns)
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"])
    sessions = pd.DatetimeIndex(pd.to_datetime(list(session_schedule(list(range(2011, 2026))).index)))
    # total returns (as P6): Yahoo adjusted before 2016-01-04, Alpaca adjusted after
    YH = p6.load_yahoo_daily(STOCKS, "2005-01-01", "2016-01-06", verbose=False)
    download_years(STOCKS, "1Day", list(range(2016, 2026)), workers=8)
    AC, _, _ = p1.load_daily(STOCKS, list(range(2016, 2026)))
    R = p6.splice_returns(YH, AC, sessions, CFG.splice)
    # split-adjusted, not dividend-adjusted prices (for capitalisation)
    YS = p7.yahoo_split_prices(STOCKS, CFG.data_start, "2016-01-06")
    AS = p7.alpaca_split_daily(STOCKS, list(range(2016, 2026)))
    cut = pd.Timestamp(CFG.splice)
    PX = AS.reindex(index=sessions, columns=R.columns)
    ys = YS.reindex(index=sessions, columns=R.columns)
    PX.loc[sessions < cut] = ys.loc[sessions < cut]
    # shares outstanding (SEC, point in time) on today's split basis
    REC = p7.sec_shares(STOCKS)
    SPL, CUR = p7.yahoo_splits_and_shares(STOCKS)
    SH, DIAG = p7.shares_daily(REC, SPL, CUR, sessions, CFG.unit_band)
    DIAG.to_csv(CACHE / "shares_diagnostics.csv", index=False)
    CAP = (PX * SH.reindex(index=sessions, columns=R.columns))
    R = R.loc[:, CAP.notna().any()]
    CAP = CAP[R.columns]
    qc = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
    sess = s05b.sample_sessions([2017, 2018, 2019], 4, seed=42) + s05b.sample_sessions(list(range(2020, 2026)), 4, seed=42)
    samples = pd.concat([pd.read_parquet(qc / f"{d}.parquet") for d in sess], ignore_index=True)
    COST = p1.daily_cost(R.index, list(R.columns), s05b.cost_model(samples))
    print(f"shares: {len(DIAG)} symbols with SEC counts (tags: {DIAG['tag'].value_counts().to_dict()}); "
          f"rescaled by the Yahoo unit check: {DIAG.loc[DIAG.rescaled, 'symbol'].tolist()}")
valid_caps = CAP.notna().sum(axis=1)
print(f"{R.shape[1]} stocks, {len(R)} sessions; stocks with a cap per year (median):")
display(valid_caps.groupby(valid_caps.index.year).median().to_frame("stocks with cap").T)
print(f"days with >= {CFG.n_ranks} caps: {(valid_caps >= CFG.n_ranks).mean():.1%}; ready in {time.time() - t0:.0f}s")

1Day: 0 batches to download (0 symbol-years), 8 parallel requests
Yahoo split-only prices: 0/503 cached; downloading 503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\e

  Alpaca split-adjusted 1Day 2016: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2017: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2018: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2019: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2020: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2021: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2022: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2023: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2024: 503 symbols downloaded
  Alpaca split-adjusted 1Day 2025: 503 symbols downloaded
  SEC shares 50/503, 20s
  SEC shares 100/503, 40s
  SEC shares 150/503, 62s
  SEC shares 200/503, 83s
  SEC shares 250/503, 104s
  SEC shares 300/503, 125s
  SEC shares 350/503, 145s
  SEC shares 400/503, 164s
  SEC shares 450/503, 184s
  SEC shares 500/503, 205s
  SEC shares 503/503, 206s
  no CIK for 2: ['PSKY', 'WBD']


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
D

  Yahoo splits/shares 50/503


D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
D

... (17 outputs left out)

D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
D:\Anaconda\envs\equity-trading\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.sort_index(inplace=True)
D

shares: 491 symbols with SEC counts (tags: {'EntityCommonStockSharesOutstanding': 467, 'WeightedAverageNumberOfSharesOutstandingBasic': 13, 'CommonStockSharesOutstanding': 11}); rescaled by the Yahoo unit check: ['ARES', 'BRK.B', 'CHTR', 'CTVA', 'ECHO', 'FOX', 'HOOD', 'IBKR', 'VMRK']
488 stocks, 3772 sessions; stocks with a cap per year (median):


,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
stocks with cap,371.0,397.0,409.0,417.0,423.0,431.0,435.0,441.0,453.0,458.0,465.0,470.0,473.0,482.0,486.0


days with >= 400 caps: 89.4%; ready in 902s


## 2. Rank-space dataset (cached)

In [3]:
t0 = time.time()
ds_path = CACHE / f"dataset_p7{p7.P7_VERSION}_K{CFG.n_ranks}.npz"
SCHED = p6.fit_schedule(R.index, CFG)
need_from = max(0, SCHED[0]["train"][0] - CFG.lookback)
if ds_path.exists():
    DS = p7.load_dataset(ds_path)
else:
    DS = p7.build_rank_dataset(R, CAP, COST, CFG, start_row=need_from)
    p7.save_dataset(DS, ds_path)
rows = np.arange(need_from, len(R) - 2)
rr, tr = DS["ret"][rows], DS["tret"][rows]
m = np.isfinite(rr) & np.isfinite(tr)
print(f"dataset {DS['x'].shape}; usable rank-days {DS['ok'][need_from:].mean():.1%}; {time.time() - t0:.0f}s")
print(f"rank return vs tradable return of the same rank: corr {np.corrcoef(rr[m], tr[m])[0, 1]:.3f}, "
      f"s.d. {np.std(rr[m]):.4f} vs {np.std(tr[m]):.4f}")
print(f"{len(SCHED)} fits per arm; first predicts from {SCHED[0]['first_day']}")

  PCA residuals 500/3294 days, 6s
  PCA residuals 1000/3294 days, 15s
  PCA residuals 1500/3294 days, 24s
  PCA residuals 2000/3294 days, 34s
  PCA residuals 2500/3294 days, 43s
  PCA residuals 3000/3294 days, 52s
dataset (3772, 400, 30); usable rank-days 93.7%; 58s
rank return vs tradable return of the same rank: corr -0.002, s.d. 31.7411 vs 0.0192
19 fits per arm; first predicts from 2017-01-03


## 3. Rolling fits, both arms

In [4]:
INFO, t_all, n_done, n_tot = [], time.time(), 0, len(ARMS) * len(SCHED)
for arm, mode in ARMS.items():
    SC = np.full(DS["ok"].shape, np.nan, np.float32)
    for j, f in enumerate(SCHED):
        tr_, va_, pr_ = (np.arange(*f[k]) for k in ("train", "valid", "predict"))
        tr_, va_ = tr_[DS["ok"][tr_].sum(1) > 0], va_[DS["ok"][va_].sum(1) > 0]
        seed_scores = []
        for s in CFG.seeds:
            p = CACHE / f"{arm}_fit{j:02d}_seed{s}.npz"
            if p.exists():
                z = np.load(p, allow_pickle=True); seed_scores.append(z["scores"]); info = json.loads(str(z["info"]))
            else:
                t1 = time.time()
                model, info = p7_model.fit(DS, tr_, va_, seed=s, mode=mode, gamma=CFG.gamma, epochs=CFG.epochs,
                                           patience=CFG.patience, lr=CFG.lr, chunk=CFG.chunk_days, device=DEVICE)
                sc = p7_model.predict_scores(model, DS, pr_, chunk=CFG.chunk_days)
                info["seconds"] = time.time() - t1
                np.savez(p, scores=sc, info=json.dumps(info)); seed_scores.append(sc)
            INFO.append({"arm": arm, "fit": j, "seed": s, "first_day": f["first_day"], **info})
        with np.errstate(invalid="ignore"):
            zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
            SC[pr_] = np.nanmean(np.stack(zs), 0)
        n_done += 1
        el = time.time() - t_all
        print(f"{arm} fit {j + 1}/{len(SCHED)} ({f['first_day']}): valid by seed "
              f"{[round(i['best_valid'], 2) for i in INFO[-len(CFG.seeds):]]}; {el / 60:.1f} min, "
              f"~{el / n_done * (n_tot - n_done) / 60:.0f} min left", flush=True)
    np.savez(CACHE / f"scores_{arm}.npz", scores=SC)
INFO = pd.DataFrame(INFO)
OUT = ROOT / "notebooks" / "Rank_Space" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
INFO.to_csv(OUT / "fits.csv", index=False, float_format="%.5g")
if not SYNTHETIC:
    DIAG.to_csv(OUT / "shares_diagnostics.csv", index=False)
display(INFO.groupby(["arm", "fit"])[["best_valid", "train", "epochs"]].mean().unstack(0).round(3))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE), "device": DEVICE,
                                               "torch": torch.__version__, "p7": p7.P7_VERSION}, indent=2))

RS-paper fit 1/19 (2017-01-03): valid by seed [0.41, 0.22, -1394.8]; 3.0 min, ~109 min left
RS-paper fit 2/19 (2017-07-03): valid by seed [-0.03, -4.77, -537.7]; 5.8 min, ~104 min left
RS-paper fit 3/19 (2017-12-29): valid by seed [0.2, 0.19, 0.19]; 10.0 min, ~116 min left
RS-paper fit 4/19 (2018-06-29): valid by seed [0.21, 0.24, 0.23]; 14.7 min, ~125 min left
RS-paper fit 5/19 (2018-12-28): valid by seed [0.24, 0.25, 0.23]; 21.7 min, ~143 min left
RS-paper fit 6/19 (2019-06-28): valid by seed [0.28, 0.28, 0.28]; 30.0 min, ~160 min left
RS-paper fit 7/19 (2019-12-26): valid by seed [0.25, 0.26, 0.27]; 35.9 min, ~159 min left
RS-paper fit 8/19 (2020-06-25): valid by seed [-114.87, -511.2, -202.59]; 37.8 min, ~142 min left
RS-paper fit 9/19 (2020-12-22): valid by seed [-524.05, -117.18, -916.51]; 39.6 min, ~128 min left
RS-paper fit 10/19 (2021-06-23): valid by seed [0.47, 0.53, 0.48]; 43.0 min, ~120 min left
RS-paper fit 11/19 (2021-12-20): valid by seed [0.31, 0.33, 0.33]; 47.9 min, ~

C:\Users\Iván\AppData\Local\Temp\ipykernel_16868\1229785172.py:21: RuntimeWarning: Mean of empty slice
  zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
D:\Anaconda\envs\equity-trading\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Iván\AppData\Local\Temp\ipykernel_16868\1229785172.py:22: RuntimeWarning: Mean of empty slice
  SC[pr_] = np.nanmean(np.stack(zs), 0)


RS-net fit 1/19 (2017-01-03): valid by seed [0.41, 1.06, 1.01]; 101.3 min, ~91 min left
RS-net fit 2/19 (2017-07-03): valid by seed [1.12, 2.3, 2.25]; 104.7 min, ~85 min left
RS-net fit 3/19 (2017-12-29): valid by seed [2.54, 2.59, 2.59]; 107.4 min, ~78 min left
RS-net fit 4/19 (2018-06-29): valid by seed [0.64, 1.48, 0.99]; 111.6 min, ~73 min left
RS-net fit 5/19 (2018-12-28): valid by seed [-1.24, 1.33, -2.29]; 116.7 min, ~68 min left
RS-net fit 6/19 (2019-06-28): valid by seed [0.2, -0.02, -0.23]; 122.1 min, ~63 min left
RS-net fit 7/19 (2019-12-26): valid by seed [-0.81, -0.23, -0.05]; 125.7 min, ~58 min left
RS-net fit 8/19 (2020-06-25): valid by seed [-0.14, 0.06, 0.23]; 127.9 min, ~52 min left
RS-net fit 9/19 (2020-12-22): valid by seed [1.37, 0.77, 1.46]; 130.4 min, ~47 min left
RS-net fit 10/19 (2021-06-23): valid by seed [-2.81, 3.05, 2.74]; 133.8 min, ~42 min left
RS-net fit 11/19 (2021-12-20): valid by seed [-2.12, 1.69, 1.64]; 136.0 min, ~36 min left
RS-net fit 12/19 (2022

best_valid           train           epochs         
arm     RS-net RS-paper RS-net RS-paper  RS-net RS-paper
fit                                                     
0        0.827 -464.721  0.375 -314.795  26.000   24.667
1        1.890 -180.834  0.687  -61.494  25.667   21.000
2        2.572    0.193  0.486 -584.534  19.000   30.000
3        1.039    0.226  0.732 -148.399  29.667   33.667
4       -0.734    0.241 -0.094 -567.876  36.333   50.000
5       -0.017    0.279  0.488   -2.379  38.333   59.667
6       -0.363    0.259  0.449  -10.485  25.333   42.333
7        0.051 -276.218  0.026 -484.096  15.000   13.000
8        1.200 -519.246  0.933    0.167  17.667   13.000
9        0.995    0.491 -0.031 -155.829  24.000   23.667
10       0.405    0.326  0.244  -11.665  15.333   35.000
11      -0.043    0.323  0.606  -28.428  46.000   91.333
12       0.488    0.219  0.603  -15.566  41.667   21.667
13      -0.110    0.151  0.221  -97.852  38.000   14.667
14      -0.754    0.201  0.366  -25.881  32.333   27.000
15      -0.992    0.277  0.626    0.216  32.000   59.667
16      -0.522    0.361  0.820    0.404  36.333   78.000
17      -0.562    0.285 -0.111    0.298  21.000   35.667
18      -1.131    0.306  0.052    0.277  26.667   33.667

186